# Smart Building Energy Forecasting — One-Hour-Ahead Load Forecast

**Task**: predict appliance energy use at `t + 60 min` from 10-minute readings
(UCI *Appliances Energy Prediction*, CC BY 4.0).

## 01 · Project overview

**Users**: energy managers, facility engineers, building-automation operators.
**Decision supported**: pre-position HVAC/load control, raise peak alerts, plan demand.
**Unit of prediction**: appliance energy (Wh) at `t + 1h` for every timestamp `t`.

**Success criteria**
1. Target shifted exactly 6 steps; no feature at `t` uses data after `t`.
2. Chronological splits + expanding-window backtest with naive/seasonal baselines.
3. Candidate model beats baseline MAE/WAPE across multiple validation folds.
4. Error analysis by hour, weekday/weekend, load quantile, and peak periods.
5. One artifact builds identical features in train and inference + a monitoring plan.

> **Design rule**: for forecasting, temporal alignment correctness matters more than
> model complexity. One step of leakage produces great scores that fail in production.

**Out of scope (10-hour timebox)**: real-time BMS integration, full probabilistic
forecasting, control-loop optimisation, multi-building generalisation.

## 02 · Environment & reproducibility

All temporal logic lives in the installable package `energy_forecasting`
(`src/`) so the notebook and any future service share one implementation —
this is the main defence against training–serving skew. Seeds only affect model
fitting; splits are deterministic from timestamps.

In [ ]:
# In Colab: clone the repo and install. Locally: install from the checkout root.
import importlib.util, subprocess, sys
if importlib.util.find_spec("energy_forecasting") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "..", "ucimlrepo"],
                   check=False)
    if importlib.util.find_spec("energy_forecasting") is None:  # editable-install fallback
        sys.path.insert(0, "../src")

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

import energy_forecasting as ef
from energy_forecasting.config import ARTIFACTS_DIR, DATA_DIR, ForecastConfig

CFG = ForecastConfig()
print("package:", ef.__version__, "| python:", platform.python_version(),
      "| sklearn:", sklearn.__version__, "| pandas:", pd.__version__)
print(CFG)

## 03 · Data ingestion

Download is separated from training: `fetch_raw` runs once, caches a CSV, and
records lineage (source URL, retrieval time, sha256). Reruns read only the cache,
so an unstable network can't corrupt an experiment.

In [ ]:
from energy_forecasting.data import fetch_raw, load_raw

fetch_raw(DATA_DIR)          # no-op if the cache already exists
raw = load_raw(DATA_DIR)     # verifies sha256 against the lineage sidecar
lineage = json.loads((DATA_DIR / "data_lineage.json").read_text())
print(raw.shape)
{k: lineage[k] for k in ("source", "license", "retrieved_at_utc", "sha256")}

## 04 · Data contract

Before any modelling we verify what the pipeline assumes: parseable monotonic
timestamps, a 10-minute modal interval, no negative energy readings. Gaps and
duplicates are *reported* (the feature builder handles them); schema or sign
violations *fail hard*.

In [ ]:
from energy_forecasting.data import validate_contract

report = validate_contract(raw, CFG)
assert report.passed, report.failures
report.to_dict()

## 05 · EDA — what structure should the model exploit?

Three questions drive the features: how strong is persistence (autocorrelation),
how strong is the daily cycle, and how heavy is the load's right tail (peaks).

In [ ]:
from energy_forecasting.data import prepare_base

base = prepare_base(raw, CFG)   # sort, dedupe, attach target_1h = shift(-6)
ser = base.set_index(CFG.timestamp_col)[CFG.target_source_col]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
ser.plot(ax=axes[0, 0], lw=0.3, title="Appliance load — full period (Wh)")
ser.iloc[:7 * 144].plot(ax=axes[0, 1], lw=0.7, title="First 7 days")
prof = ser.groupby(ser.index.hour).mean()
prof.plot(ax=axes[1, 0], marker="o", title="Mean load by hour of day")
ser.hist(bins=60, ax=axes[1, 1])
axes[1, 1].set_title("Load distribution (right-skewed peaks)")
plt.tight_layout()

In [ ]:
acf = {k: round(ser.autocorr(lag=k), 3) for k in (1, 3, 6, 12, 144)}
print("Autocorrelation:", acf)
print("=> strong persistence at short lags and a daily (lag-144) cycle:")
print("   this motivates lag_1/6/12, rolling stats, and a seasonal lag_144 feature.")

## 06 · Leakage analysis — the availability audit

Timeline for one prediction at time `t`:

```
 ── lag_144 ─── lag_12 · lag_6 · lag_1 ── t ──────────► t+6 (target, +60 min)
    [────── rolling windows (shift 1) ──]│
             everything the model sees ──┘        never crosses this line
```

Every feature is audited against the rule *known at t*. Traps this design avoids:
random shuffling, centered rolling windows, un-shifted rolling means, and weather
observed after `t`.

In [ ]:
from energy_forecasting.features import add_features, availability_audit

frame, feature_cols = add_features(base, CFG)
audit = availability_audit(feature_cols, CFG)
assert not audit["uses_future"].any()
print(f"{len(feature_cols)} features; rows dropped for insufficient history:",
      frame.attrs["rows_dropped_insufficient_history"])
audit.head(15)

**Conclusion**: all features map to `t` or earlier. The unit tests in
`tests/test_temporal_features.py` additionally *mutate the future* and assert
features at `t` are unchanged — leakage protection is enforced by CI, not by
convention.

## 07 · Split strategy — chronological + expanding-window backtest

70/15/15 by time. Inside train+val, three expanding-window folds simulate
"train on the past, forecast the next block". The invariant
`max(train) < min(val) < min(test)` is asserted, not assumed. The test block is
untouched until section 12.

In [ ]:
from energy_forecasting.splits import (assert_temporal_order, backtest_calendar,
                                         chronological_split, expanding_window_folds)

X, y, ts = frame[feature_cols], frame[CFG.target_col], frame[CFG.timestamp_col]
split = chronological_split(len(frame), CFG)
assert_temporal_order(ts, split)

n_trainval = len(split.train) + len(split.val)
folds = expanding_window_folds(n_trainval, CFG)
backtest_calendar(ts.iloc[:n_trainval], folds)

## 08 · Preprocessing as a reusable transform

`add_features` **is** the preprocessing artifact — inference calls the same
function on raw history (see section 17), so there is no second implementation
to drift. Scaling for the linear model lives inside its Pipeline and is fit
per-fold, never on validation data.

In [ ]:
X.describe().loc[["mean", "std", "min", "max"],
                 ["load_now", "lag_1", "lag_144", "roll_mean_6", "seasonal_ref"]].round(1)

## 09 · Naive baselines — the bar to clear

*Last value* (persistence) and *seasonal naive* (same time yesterday, aligned to
the **target** timestamp) run through the exact same backtest loop as the real
models. A model that can't beat them consistently has no added value.

In [ ]:
from energy_forecasting.backtest import run_backtest, summarize_backtest
from energy_forecasting.models import model_factories

factories = model_factories(CFG)
baseline_results = run_backtest(
    X.iloc[:n_trainval], y.iloc[:n_trainval], ts.iloc[:n_trainval], folds,
    {k: factories[k] for k in ("last_value", "seasonal_naive")}, CFG)
summarize_backtest(baseline_results).round(2)

## 10 · Candidate models

- **Ridge** (scaled, in-Pipeline): interpretable linear reference.
- **HistGradientBoosting** (MAE loss): primary nonlinear candidate — captures
  interactions at CPU-friendly cost.

No LSTM/Transformer: with ~19k rows and a 10-hour timebox, proving temporal
correctness and stability is worth more than sequence-model complexity.

In [ ]:
candidate_results = run_backtest(
    X.iloc[:n_trainval], y.iloc[:n_trainval], ts.iloc[:n_trainval], folds,
    {k: factories[k] for k in ("ridge", "hgb")}, CFG)
results = pd.concat([baseline_results, candidate_results], ignore_index=True)
results.round(2)

## 11 · Model comparison & selection

Selection rule (fixed *before* looking at test data): eligible only if the
candidate beats **every** baseline's MAE in a majority of folds; among eligible,
lowest mean MAE wins, near-ties (<2 %) break toward lower peak MAE. Latency and
size are reported because operability matters.

In [ ]:
from energy_forecasting.backtest import select_model

summary = summarize_backtest(results)
display(summary.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
for name, grp in results.groupby("model"):
    ax.plot(grp["fold"], grp["mae"], marker="o", label=name)
ax.set(xlabel="backtest fold", ylabel="MAE (Wh)", title="Stability across folds",
       xticks=sorted(results["fold"].unique()))
ax.legend()

selection = select_model(results)
print("SELECTED:", selection["selected"], "—", selection["reason"])

## 12 · Final evaluation — the test set is opened **once**

Design frozen (features, folds, selection rule) → refit the winner on train+val →
score the held-out final block a single time. The peak threshold (train-only
q90) travels with the evaluation.

In [ ]:
from energy_forecasting.metrics import evaluate_all

final_model = factories[selection["selected"]]()
final_model.fit(X.iloc[:n_trainval], y.iloc[:n_trainval])

peak_thr = float(y.iloc[:n_trainval].quantile(CFG.peak_quantile))
y_test = y.iloc[split.test].to_numpy()
ts_test = ts.iloc[split.test]
test_pred = final_model.predict(X.iloc[split.test])

test_metrics = evaluate_all(y_test, test_pred, peak_thr)
test_metrics

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
show = slice(0, 4 * 144)  # first 4 test days
ax.plot(ts_test.iloc[show], y_test[show], lw=0.8, label="actual")
ax.plot(ts_test.iloc[show], test_pred[show], lw=0.8, label="forecast")
ax.axhline(peak_thr, ls="--", c="red", lw=0.8, label=f"peak threshold (train q90={peak_thr:.0f})")
ax.set_title("Actual vs 1h-ahead forecast — first 4 test days")
ax.legend()

## 13 · Error analysis — where does it fail?

Averages hide the failures that matter for load planning. We slice by hour,
day type, and train-defined load quartile, and check residual autocorrelation
(structure left in the residuals = signal the model missed).

In [ ]:
from energy_forecasting.metrics import residual_summary, slice_errors

quartiles = [float(y.iloc[:n_trainval].quantile(q)) for q in (0.25, 0.5, 0.75)]
slices = slice_errors(ts_test, y_test, test_pred, quartiles)

fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))
for ax, dim in zip(axes, ("hour", "day_type", "load_quartile")):
    sub = slices[slices["slice_dim"] == dim]
    ax.bar(sub["slice_value"].astype(str), sub["mae"])
    ax.set_title(f"MAE by {dim}")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

resid = residual_summary(ts_test, y_test, test_pred, peak_thr)
print("bias:", round(resid["metrics"]["bias"], 2),
      "| residual ACF:", resid["residual_autocorrelation"])

## 14 · Robustness — degraded inputs must degrade gracefully

Production feeds go stale and lose sensors. The inference contract (section 17)
returns a *flagged fallback* instead of feeding NaN to the model. Here we
exercise those paths end-to-end.

In [ ]:
from energy_forecasting.inference import ForecastBundle, predict_one
from datetime import datetime, timezone

bundle = ForecastBundle(model=final_model, feature_columns=feature_cols, config=CFG,
                        model_version=CFG.model_version,
                        trained_at=datetime.now(timezone.utc).isoformat())

t_ok = base[CFG.timestamp_col].iloc[-1]
ok = predict_one(bundle, raw, t_ok)

stale = predict_one(bundle, raw, t_ok + pd.Timedelta(hours=2))       # feed died
short = predict_one(bundle, raw.tail(50), raw["date"].iloc[-1])      # history too short

pd.DataFrame([ok, stale, short])[["quality_flag", "fallback_used", "forecast_appliances_wh"]]

## 15 · Explainability & negative controls

Permutation importance on the validation tail. Expectation: recent-load features
dominate; the random columns `rv1`/`rv2` sit at ~0. If a noise column ranked
high, the model would be fitting noise — an automatic red flag.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(final_model, X.iloc[split.val], y.iloc[split.val],
                              n_repeats=5, random_state=CFG.seed,
                              scoring="neg_mean_absolute_error")
imp = (pd.Series(perm.importances_mean, index=feature_cols)
       .sort_values(ascending=False))
display(imp.head(10).round(3).to_frame("MAE increase when permuted"))
print("negative controls:", imp[["rv1", "rv2"]].round(4).to_dict())

## 16 · Packaging

`python -m energy_forecasting.train` reproduces everything headlessly and writes
the artifact set (bundle + sha256, feature config, backtest/selection/residual
reports). Here we save the same bundle from the notebook run.

In [ ]:
from energy_forecasting.inference import save_bundle

bundle.metrics = {"selected_model": selection["selected"], "test": test_metrics}
bundle_path = save_bundle(bundle, ARTIFACTS_DIR)
CFG.save_json(ARTIFACTS_DIR / "feature_config.json",
              extra={"feature_columns": feature_cols,
                     "selected_model": selection["selected"]})
results.to_csv(ARTIFACTS_DIR / "backtest_metrics.csv", index=False)
sorted(p.name for p in ARTIFACTS_DIR.iterdir())

## 17 · Inference demo — cold start from disk

Reload the bundle (sha256 verified before unpickling) and forecast from raw
history only, proving the artifact is self-contained and reproducible.

In [ ]:
from energy_forecasting.inference import load_bundle

reloaded = load_bundle(bundle_path)
demo = predict_one(reloaded, raw, t_ok)
assert demo["forecast_appliances_wh"] == ok["forecast_appliances_wh"]  # identity check
demo

## 18 · Monitoring & retraining design

| Dimension | Monitor | Action on anomaly |
|---|---|---|
| **Freshness** | minutes since last sensor timestamp | fallback forecast + alert data pipeline |
| **Data quality** | gaps, duplicates, impossible ranges | quarantine / impute per policy |
| **Drift** | load & weather distributions vs train window | re-evaluate on recent rolling window |
| **Forecast** | prediction distribution, bias proxy | investigate systematic under/over-forecast |
| **Performance** | MAE/WAPE/peak-MAE as actuals arrive (60 min lag) | retrain / recalibrate past threshold |

Operating notes: schedule on **event time** (not processing time) to absorb
delayed readings; keep forecasting decoupled from automatic control — operator
confirms setpoint changes, with safe bounds, manual override, and an audit log.

**Retraining triggers**: rolling 7-day MAE > 1.2× backtest mean · bias outside
±10 Wh for 3 days · a schedule/season change flagged by the facility team.

## 19 · Limitations — what this project does *not* prove

1. **Domain**: one Belgian house, ~4.5 months. Thai buildings (tropical seasons,
   different holidays, occupancy, BMS) require re-validation and retraining.
2. **Point forecasts only** — no uncertainty intervals (stretch: quantile /
   conformal regression).
3. **Weather observed at `t`**, not a forecast feed; fine at 1 h, not day-ahead.
4. **Offline backtest** — no streaming latency, sensor outages, or concept drift
   beyond the recorded period.
5. **No causal/economic claim**: proven result = forecast accuracy on held-out
   time under stated availability assumptions — not energy-cost savings.

**Dataset attribution**: Candanedo, L. (2017). *Appliances Energy Prediction*
[Dataset]. UCI Machine Learning Repository — CC BY 4.0.